# Credit risk EDA
Run on official cs-training.csv only. All observations are retained. Descriptive EDA uses the raw dataset; future model decisions must use training data and keep test evaluation locked.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / 'configs/default.yaml').exists() and (ROOT.parent / 'configs/default.yaml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.config import load_config, resolve
from src.data import load_data, FEATURES, quality_report
cfg = load_config()
df = load_data()
target = cfg['target']
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
figures = resolve(cfg['outputs']['figures']); figures.mkdir(parents=True, exist_ok=True)
tables = resolve(cfg['outputs']['tables']); tables.mkdir(parents=True, exist_ok=True)
def table(name, value):
    value.to_csv(tables / f'{name}.csv')
    display(value)
def figure(name):
    plt.tight_layout()
    plt.savefig(figures / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()


## Overview
Inspect dimensions, types, descriptive statistics, duplicate counts, label imbalance and missingness before deciding any cleaning policy.

In [ ]:
print('Dimensions:', df.shape)
print('Quality:', quality_report(df))
table('dtypes', df.dtypes.to_frame('dtype'))
table('descriptive_statistics', df.describe().T)
counts = df[target].value_counts().sort_index()
table('target_distribution', pd.DataFrame({'count': counts, 'proportion': counts/len(df)}))
print('Majority/minority ratio:', counts.max()/counts.min())
table('missingness', pd.DataFrame({'count':df.isna().sum(),'percent':df.isna().mean()*100}))
counts.plot.bar(title='Serious delinquency labels'); figure('target_distribution')


## Distributions and outliers
Histograms and boxplots retain extremes. The IQR rule flags candidates for investigation; it does not establish invalidity.

In [ ]:
df[FEATURES].hist(bins=40, figsize=(16,12)); figure('feature_histograms')
fig, axes = plt.subplots(5,2,figsize=(16,18))
for col, ax in zip(FEATURES, axes.flat):
    sns.boxplot(x=df[col], ax=ax); ax.set_title(col)
figure('feature_boxplots')
q1, q3 = df[FEATURES].quantile(.25), df[FEATURES].quantile(.75)
iqr = q3-q1
flags = ((df[FEATURES] < q1-1.5*iqr) | (df[FEATURES] > q3+1.5*iqr)).sum()
table('iqr_flags', flags.to_frame('candidate_count'))


## Correlation and class comparisons
Pearson correlations describe association and can be sensitive to extremes. Class-normalized histograms help compare distributions despite imbalance.

In [ ]:
plt.figure(figsize=(13,10))
corr = df[FEATURES+[target]].corr()
sns.heatmap(corr, cmap='coolwarm', center=0)
figure('correlation_heatmap'); table('correlations', corr)
fig, axes = plt.subplots(5,2,figsize=(16,20))
for col, ax in zip(FEATURES, axes.flat):
    sns.histplot(data=df, x=col, hue=target, bins=30, stat='density', common_norm=False, element='step', ax=ax)
figure('distributions_by_target')


## Suspicious values
Review large delinquency counts (including 96 and 98), debt ratios and revolving utilization above one, age zero, and negative feature values. These are investigation flags, not proof of invalid records. Check the official dictionary and domain interpretation before changing them.

In [ ]:
checks = {}
for col in [c for c in FEATURES if 'PastDue' in c or '90DaysLate' in c]:
    checks[col+'_96_or_98'] = int(df[col].isin([96,98]).sum())
    table(col+'_value_counts', df[col].value_counts(dropna=False).sort_index().to_frame('count'))
for col in ['DebtRatio','RevolvingUtilizationOfUnsecuredLines']:
    checks[col+'_above_one'] = int((df[col]>1).sum())
    table(col+'_quantiles', df[col].quantile([0,.5,.9,.95,.99,1]).to_frame('value'))
checks['age_zero'] = int((df['age']==0).sum())
table('investigation_flags', pd.Series(checks).to_frame('count'))
table('negative_values', (df[FEATURES]<0).sum().to_frame('count'))


## Review checklist
Update docs/eda_findings.md with measured results and justified interpretation. Agree any treatment using training data only. No fitted preprocessing or final model is produced by this notebook.